# 增量语义 Gaussian：L4 复现实验

选择 NVIDIA L4。使用官方真实数据与已归档的 SAM/CLIP 教师、前缀图缓存。上传本项目的 V1 或 V2 结果 ZIP 后，只恢复输入缓存，重新训练所有比较，不使用归档训练结果作为完成标记。

2 个开发场景 × 8 组合 × 2 种种子，另有 TUM desk2 独立录像验证。区域指标是 SAM 候选区域一致性，非人工语义/部件真值准确率。已知位姿建图，非完整 SLAM。计算耗时及 CUDA 数值可能因运行时版本改变，不能保证跨 GPU 位级重现。

In [ ]:
#@title Unpack frozen project source
from pathlib import Path
import io,zipfile,base64,os,sys,subprocess,json,hashlib
ROOT=Path('/content/semantic-incremental-v2');ROOT.mkdir(exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(ROOT)
os.chdir(ROOT)
print('SOURCE_READY',ROOT)


In [ ]:
#@title Upload the archived experiment ZIP and restore only inputs
from google.colab import files
uploaded=files.upload()
assert len(uploaded)==1,'Choose one verified V1 or V2 experiment ZIP.'
archive=next(iter(uploaded.values()));INPUT=Path('/content/semantic-incremental');INPUT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
 assert z.testzip() is None
 for entry in z.infolist():
  parts=Path(entry.filename).parts
  if '..' in parts:raise ValueError('Unexpected archive path')
  if len(parts)>2 and parts[1]=='data' and not entry.is_dir():
   dest=INPUT.joinpath(*parts[1:]);dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(entry))
  if len(parts)==2 and parts[1]=='experiment-plan.json':
   (INPUT/'experiment-plan.json').write_bytes(z.read(entry))
assert (INPUT/'data/tum-desk/prefixes/prefix-6.npz').exists()
print('FROZEN_CACHES_READY; archived trained maps have not been copied into results/')


In [ ]:
#@title Install official dependencies and download public posed imagery
import torch
assert torch.cuda.is_available(),'Choose L4 before running.'
print('GPU',torch.cuda.get_device_name())
subprocess.run([sys.executable,'bootstrap_colab.py'],cwd=ROOT,check=True)
subprocess.run([sys.executable,'prepare_l4.py'],cwd=ROOT,check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,check=True)
subprocess.run([sys.executable,'-m','hglab.gpu_smoke'],cwd=ROOT,check=True)


In [ ]:
#@title Start the screen and guarded continuation
import psutil
active=[p.pid for p in psutil.process_iter(['cmdline','cwd']) if p.info['cwd']==str(ROOT) and any('run_iteration.py' in x or 'coordinate.py' in x for x in p.info['cmdline'] or [])]
assert not active,('Experiment already active',active)
assert not (ROOT/'status.json').exists(),'Use a fresh directory for a new protocol; inspect existing status before resuming.'
(ROOT/'status.json').write_text(json.dumps(dict(state='starting',phase='screen',completed=[],total=4)))
screen=subprocess.Popen([sys.executable,'run_iteration.py','--phase','screen'],cwd=ROOT,stdout=open('/content/l4-screen.log','w'),stderr=subprocess.STDOUT,start_new_session=True)
coordinator=subprocess.Popen([sys.executable,'coordinate.py'],cwd=ROOT,stdout=open('/content/l4-coordinate.log','w'),stderr=subprocess.STDOUT,start_new_session=True)
monitor=subprocess.Popen([sys.executable,'monitor_memory.py'],cwd=ROOT,stdout=open('/content/l4-monitor.log','w'),stderr=subprocess.STDOUT,start_new_session=True)
print('SCREEN',screen.pid,'COORDINATOR',coordinator.pid)


In [ ]:
#@title Inspect results without hiding failures
print((ROOT/'status.json').read_text())
for name in ['screen-summary.json','confirmation-summary.json','attention-needed.json','coordinator-failure.txt','finish-status.json']:
 p=ROOT/name
 if p.exists():print(name,p.read_text()[-5000:])
print(Path('/content/l4-coordinate.log').read_text()[-3000:])


In [ ]:
#@title Download the completed evidence archive
from google.colab import files
status=json.loads((ROOT/'finish-status.json').read_text());assert status['state']=='complete'
files.download(status['artifact'])
